# Auto-Registrierung Schritt-fuer-Schritt

Dieses Notebook zeigt die automatische Registrierung in klaren Einzelschritten:
1. Mesh laden
2. Boundary-Loops inspizieren
3. Face-Graph-Normalen-Cluster als Opening-Kandidaten
4. Finale Auto-Openings
5. Centerline, Endpunkte, Bifurkation visualisieren


In [1]:
from pathlib import Path
from types import SimpleNamespace
import sys
import copy

import numpy as np
import plotly.graph_objects as go

# Repo root finden (Ordner, der 'ghd' enthaelt)
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'ghd').is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'ghd').is_dir():
    raise RuntimeError("Could not locate repo root containing 'ghd'.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ghd.fitting.registration import RegistrationwOpeningAlignmentwDifferentiableCentreline


In [2]:
# ----------------------
# User config
# ----------------------
CASE_ROOT = REPO_ROOT / 'checkpoints/alignment'
CASE_NAME = 'C0029'

NUM_OPENINGS = 3
NUM_CEP = 3
AUTO_MIN_LOOP_VERTICES = 16
AUTO_CAST_STEP_SIZE = 2
AUTO_NORMAL_DOT_MIN = 0.84
AUTO_FACE_DOT_MIN = 0.84

MAX_CANDIDATES_TO_SHOW = 12

case_dir = CASE_ROOT / CASE_NAME
assert case_dir.exists(), f'Case folder not found: {case_dir}'
print('Case:', case_dir)


Case: /workspace/AneuG/checkpoints/alignment/C0029


In [3]:
def safe_indices(idx, n):
    idx = np.asarray(idx, dtype=np.int64).reshape(-1)
    if idx.size == 0:
        return idx
    return idx[(idx >= 0) & (idx < n)]

def add_mesh(fig, verts, faces, color='lightgray', opacity=0.12):
    fig.add_trace(go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=color, opacity=opacity, hoverinfo='skip', name='mesh', showlegend=False
    ))

def add_loop(fig, loop_xyz, name, color, width=4, showlegend=False):
    if loop_xyz.shape[0] < 2:
        return
    closed = np.vstack([loop_xyz, loop_xyz[0]])
    fig.add_trace(go.Scatter3d(
        x=closed[:, 0], y=closed[:, 1], z=closed[:, 2],
        mode='lines', line=dict(color=color, width=width),
        name=name, showlegend=showlegend
    ))

def add_points(fig, xyz, name, color, size=5, symbol='circle', showlegend=False, text=None):
    if xyz.shape[0] == 0:
        return
    fig.add_trace(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode='markers+text' if text is not None else 'markers',
        marker=dict(size=size, color=color, symbol=symbol),
        text=text, textposition='top center',
        name=name, showlegend=showlegend
    ))

def add_opening_surfaces_from_reg(fig, reg, color='deepskyblue', opacity=0.45):
    for i, (v, f) in enumerate(zip(getattr(reg, 'op_rec_v', []), getattr(reg, 'op_rec_f', []))):
        v = np.asarray(v, dtype=np.float64).reshape(-1, 3)
        f = np.asarray(f, dtype=np.int64).reshape(-1, 3)
        if v.shape[0] < 3 or f.shape[0] < 1:
            continue
        fig.add_trace(go.Mesh3d(
            x=v[:, 0], y=v[:, 1], z=v[:, 2],
            i=f[:, 0], j=f[:, 1], k=f[:, 2],
            color=color, opacity=opacity, hoverinfo='skip',
            name='auto opening patch', showlegend=(i == 0)
        ))

def centerline_paths_from_reg(reg, verts):
    paths = getattr(reg, 'centreline_branch_paths', None)
    if paths is None:
        return []
    polylines = []
    for p in paths:
        idx = safe_indices(p, len(verts))
        if idx.size >= 2:
            polylines.append(verts[idx])
    return polylines


In [4]:
# Initialisiere Registrierungsobjekt (nur laden, noch keine Auto-Registration)
args = SimpleNamespace(device='cpu')
reg = RegistrationwOpeningAlignmentwDifferentiableCentreline(
    args=args,
    root=str(CASE_ROOT),
    target=CASE_NAME,
    num_op=NUM_OPENINGS,
    num_cep=NUM_CEP,
    step_size=AUTO_CAST_STEP_SIZE,
)

verts = np.asarray(reg.mesh_target.vertices)
faces = np.asarray(reg.mesh_target.triangles)
print('verts:', verts.shape, 'faces:', faces.shape)


Loading mesh from: /workspace/AneuG/checkpoints/alignment/C0029/part_aligned.obj
verts: (16154, 3) faces: (32304, 3)


In [5]:
# Schritt 1: Grundmesh visualisieren
fig = go.Figure()
add_mesh(fig, verts, faces, color='lightgray', opacity=0.14)
fig.update_layout(title=f'Base mesh | {CASE_NAME}', scene=dict(aspectmode='data'), width=1050, height=760)
fig.show()


In [6]:
# Schritt 2: Boundary-Loops (falls vorhanden)
boundary_loops = reg._extract_boundary_loops(min_loop_vertices=max(6, AUTO_MIN_LOOP_VERTICES // 2))
print('boundary loops:', len(boundary_loops))

fig = go.Figure()
add_mesh(fig, verts, faces, color='lightgray', opacity=0.10)
palette = ['orange', 'red', 'gold', 'green', 'purple', 'brown', 'magenta', 'cyan']
for i, loop_idx in enumerate(boundary_loops):
    idx = safe_indices(loop_idx, len(verts))
    if idx.size < 3:
        continue
    xyz = verts[idx]
    color = palette[i % len(palette)]
    add_loop(fig, xyz, name=f'boundary loop {i}', color=color, width=5, showlegend=True)
    c = xyz.mean(axis=0, keepdims=True)
    add_points(fig, c, name=f'boundary center {i}', color=color, size=4, showlegend=False)

fig.update_layout(title=f'Step 2: Boundary loops | {CASE_NAME}', scene=dict(aspectmode='data'), width=1150, height=780)
fig.show()


boundary loops: 0


## Detail-Trace der Pipeline (1-9)

Die folgenden Zellen zeigen die internen Zwischenschritte aus `register_openings_auto_normals` explizit:
- Reset + Debug-Init
- Face-Adjacency-Graph und Dot-Produkt-Filter
- Connected Components auf Faces
- Boundary-Edges (Count=1) und Loop-Komponenten
- Loop-Metriken/Score
- Multi-Threshold Kandidaten, Deduplizierung, Greedy-Selection
- Endpoint-Mapping + Centerline + Repair-Status


In [7]:
# Step 1: Reset + Debug-Init explizit
print('Before reset:', len(reg.op_v_indices), len(reg.op_v_coords), len(reg.op_rec_v), len(reg.op_cut_points))
reg._reset_opening_state()
reg.auto_registration_debug = {
    'method': 'register_openings_auto_normals',
    'fallback_from': None,
    'fallback_reason': None,
    'min_loop_vertices': int(AUTO_MIN_LOOP_VERTICES),
    'normal_dot_min': float(AUTO_NORMAL_DOT_MIN),
    'face_dot_min': float(AUTO_FACE_DOT_MIN),
}
print('After reset:', len(reg.op_v_indices), len(reg.op_v_coords), len(reg.op_rec_v), len(reg.op_cut_points))
reg.auto_registration_debug


Before reset: 0 0 0 0
After reset: 0 0 0 0


{'method': 'register_openings_auto_normals',
 'fallback_from': None,
 'fallback_reason': None,
 'min_loop_vertices': 16,
 'normal_dot_min': 0.84,
 'face_dot_min': 0.84}

In [8]:
# Step 2: faces, face_normals, face_adjacency + Dot-Produkt
import igraph as ig

mesh = reg.mesh_target_trimesh
faces_m = np.asarray(mesh.faces, dtype=np.int64)
face_normals = np.asarray(mesh.face_normals, dtype=np.float64)
face_adj = np.asarray(mesh.face_adjacency, dtype=np.int64)

ndot = np.abs(np.einsum('ij,ij->i', face_normals[face_adj[:, 0]], face_normals[face_adj[:, 1]]))
thr = float(max(AUTO_NORMAL_DOT_MIN, AUTO_FACE_DOT_MIN))
keep = ndot >= thr
kept_edges = face_adj[keep]

print('faces:', faces_m.shape, 'face_normals:', face_normals.shape, 'face_adj:', face_adj.shape)
print('ndot stats min/mean/max:', float(ndot.min()), float(ndot.mean()), float(ndot.max()))
print(f'keep threshold={thr:.3f}: kept {kept_edges.shape[0]} / {face_adj.shape[0]} adjacency edges')


faces: (32304, 3) face_normals: (32304, 3) face_adj: (48456, 2)
ndot stats min/mean/max: 0.0010744876318090069 0.9884951110779601 1.0
keep threshold=0.840: kept 47888 / 48456 adjacency edges


In [9]:




# Step 2b: Connected Components im Face-Graph
if kept_edges.shape[0] == 0:
    raise RuntimeError('No kept face-adjacency edges for current threshold.')

fgraph = ig.Graph(n=faces_m.shape[0], edges=kept_edges.tolist(), directed=False)
fcomps = fgraph.components()
comp_sizes = np.array([len(c) for c in fcomps], dtype=np.int64)
print('num face-components:', len(fcomps))
print('largest sizes:', sorted(comp_sizes.tolist(), reverse=True)[:12])


num face-components: 5
largest sizes: [27000, 2359, 1467, 949, 529]


In [10]:
# Step 2c: Face-Graph (gefilterte Nachbarschaften) in 3D visualisieren
face_centroids = np.mean(verts[faces_m], axis=1)
sample_n = min(3500, kept_edges.shape[0])
if sample_n < kept_edges.shape[0]:
    rng = np.random.default_rng(7)
    sel = rng.choice(kept_edges.shape[0], size=sample_n, replace=False)
    draw_edges = kept_edges[sel]
else:
    draw_edges = kept_edges

xe, ye, ze = [], [], []
for a, b in draw_edges:
    pa = face_centroids[int(a)]
    pb = face_centroids[int(b)]
    xe.extend([pa[0], pb[0], None])
    ye.extend([pa[1], pb[1], None])
    ze.extend([pa[2], pb[2], None])

fig = go.Figure()
add_mesh(fig, verts, faces, color='lightgray', opacity=0.08)
fig.add_trace(go.Scatter3d(
    x=xe, y=ye, z=ze, mode='lines',
    line=dict(color='royalblue', width=1),
    name='filtered face-adjacency edges'
))
fig.update_layout(
    title=f'Step 2c: Face-Graph edges after dot-filter (sample={draw_edges.shape[0]})',
    scene=dict(aspectmode='data'), width=1200, height=800
)
fig.show()


In [12]:
# Step 3: Aus einer Face-Component Boundary-Edges (Count=1) und Loop-Komponenten ableiten
comp_face_min = max(6, int(AUTO_MIN_LOOP_VERTICES) // 2)
valid_comps = [np.asarray(c, dtype=np.int64) for c in fcomps if len(c) >= comp_face_min]
if len(valid_comps) == 0:
    raise RuntimeError('No valid face components for boundary extraction.')

comp0 = valid_comps[int(np.argmax([len(c) for c in valid_comps]))]
comp_faces = faces_m[comp0]
tri_edges = np.concatenate((comp_faces[:, [0, 1]], comp_faces[:, [1, 2]], comp_faces[:, [2, 0]]), axis=0)
tri_edges = np.sort(tri_edges, axis=1)
uniq_edges, counts = np.unique(tri_edges, axis=0, return_counts=True)
boundary_edges = uniq_edges[counts == 1]
boundary_vertices = np.unique(boundary_edges.reshape(-1)) if boundary_edges.size else np.array([], dtype=np.int64)

print('selected component faces:', comp0.size)
print('all tri edges in component:', tri_edges.shape[0])
print('boundary edges (count=1):', boundary_edges.shape[0])
print('boundary vertices:', boundary_vertices.shape[0])

if boundary_vertices.size > 0:
    lut = -np.ones(verts.shape[0], dtype=np.int64)
    lut[boundary_vertices] = np.arange(boundary_vertices.shape[0], dtype=np.int64)
    local_edges = lut[boundary_edges]
    bgraph = ig.Graph(n=boundary_vertices.shape[0], edges=local_edges.tolist(), directed=False)
    bcomps = bgraph.components()
    bcomp_sizes = sorted([len(c) for c in bcomps], reverse=True)
    print('boundary connected components:', len(bcomps), 'largest:', bcomp_sizes[:10])
else:
    bcomps = []


selected component faces: 27000
all tri edges in component: 81000
boundary edges (count=1): 568
boundary vertices: 568
boundary connected components: 4 largest: [219, 149, 111, 89]


In [13]:
# Step 3b: Boundary-Edges und groesste Boundary-Loops visualisieren
fig = go.Figure()
add_mesh(fig, verts, faces, color='lightgray', opacity=0.08)

if boundary_edges.size > 0:
    xe, ye, ze = [], [], []
    for a, b in boundary_edges[: min(5000, boundary_edges.shape[0])]:
        pa = verts[int(a)]
        pb = verts[int(b)]
        xe.extend([pa[0], pb[0], None])
        ye.extend([pa[1], pb[1], None])
        ze.extend([pa[2], pb[2], None])
    fig.add_trace(go.Scatter3d(
        x=xe, y=ye, z=ze, mode='lines',
        line=dict(color='orange', width=2), name='boundary edges (count=1)'
    ))

    if len(bcomps) > 0:
        order = np.argsort([len(c) for c in bcomps])[::-1]
        for rank, k in enumerate(order[:6]):
            comp_idx = boundary_vertices[np.asarray(bcomps[int(k)], dtype=np.int64)]
            xyz = verts[comp_idx]
            add_loop(fig, xyz, name=f'loop comp {rank}', color='deepskyblue', width=4, showlegend=(rank==0))

fig.update_layout(title='Step 3b: Boundary edges + loop components', scene=dict(aspectmode='data'), width=1200, height=800)
fig.show()


In [14]:
# Step 4: Loop-Metriken/Score fuer Cluster-Kandidaten explizit auflisten
cand_all = reg._extract_normal_graph_cluster_candidates(
    normal_dot_min=float(thr),
    min_loop_vertices=int(AUTO_MIN_LOOP_VERTICES),
)
cand_all = sorted(cand_all, key=lambda x: float(x.get('score', 1e9)))
print('candidates at current thr:', len(cand_all))

for i, cand in enumerate(cand_all[:12]):
    idx = safe_indices(cand['loop_idx'], len(verts))
    if idx.size < 3:
        continue
    coords = verts[idx]
    center = np.mean(coords, axis=0)
    normal = reg._estimate_loop_normal(coords)
    radial = np.linalg.norm(coords - center.reshape(1, 3), axis=1)
    coverage = reg._loop_angular_coverage(coords, normal, bins=18)
    centered = coords - center.reshape(1, 3)
    try:
        _, svals, _ = np.linalg.svd(centered, full_matrices=False)
        planarity_pen = 2.2 * float(svals[-1] / (svals[0] + 1e-6))
    except Exception:
        planarity_pen = np.nan
    circ_pen = 0.25 * float(np.std(radial) / (np.mean(radial) + 1e-6))
    size_pen = max(0.0, float(AUTO_MIN_LOOP_VERTICES - idx.size)) * 0.05
    print(
        f'cand {i:02d} | n={idx.size:4d} | coverage={coverage:.3f} | '
        f'planarity_pen={planarity_pen:.3f} | circ_pen={circ_pen:.3f} | '
        f'size_pen={size_pen:.3f} | score={float(cand["score"]):.3f}'
    )


candidates at current thr: 8
cand 00 | n= 149 | coverage=1.000 | planarity_pen=0.000 | circ_pen=0.016 | size_pen=0.000 | score=-4.006
cand 01 | n= 149 | coverage=1.000 | planarity_pen=0.000 | circ_pen=0.016 | size_pen=0.000 | score=-4.006
cand 02 | n= 111 | coverage=1.000 | planarity_pen=0.000 | circ_pen=0.012 | size_pen=0.000 | score=-3.862
cand 03 | n= 111 | coverage=1.000 | planarity_pen=0.000 | circ_pen=0.012 | size_pen=0.000 | score=-3.862
cand 04 | n=  89 | coverage=1.000 | planarity_pen=0.000 | circ_pen=0.009 | size_pen=0.000 | score=-2.743
cand 05 | n=  89 | coverage=1.000 | planarity_pen=0.000 | circ_pen=0.009 | size_pen=0.000 | score=-2.743
cand 06 | n= 219 | coverage=1.000 | planarity_pen=0.085 | circ_pen=0.079 | size_pen=0.000 | score=-2.010
cand 07 | n= 219 | coverage=1.000 | planarity_pen=0.085 | circ_pen=0.079 | size_pen=0.000 | score=-2.010


In [15]:
# Step 5 + 6 + 7: Multi-threshold Kandidaten, Deduplizierung, Greedy-Selection explizit
base_thr = float(max(AUTO_NORMAL_DOT_MIN, AUTO_FACE_DOT_MIN))
thresholds = [base_thr, max(0.82, base_thr - 0.06), max(0.75, base_thr - 0.12)]
raw = []
for t in thresholds:
    c = reg._extract_normal_graph_cluster_candidates(float(t), int(AUTO_MIN_LOOP_VERTICES))
    print(f'thr={t:.3f} -> {len(c)} raw candidates')
    raw.extend(c)

# Deduplizierung wie im Code
cache = reg._get_mesh_graph_cache()
median_edge = float(cache['median_edge_length'])
dedup = []
for cand in raw:
    loop_idx = np.asarray(cand['loop_idx'], dtype=np.int64)
    keep = True
    for j, prev in enumerate(dedup):
        inter = np.intersect1d(loop_idx, np.asarray(prev['loop_idx'], dtype=np.int64)).size
        denom = max(1, min(loop_idx.size, len(prev['loop_idx'])))
        overlap = inter / float(denom)
        center_dist = np.linalg.norm(np.asarray(cand['center']) - np.asarray(prev['center']))
        if overlap > 0.65 or center_dist < 2.0 * median_edge:
            if loop_idx.size > len(prev['loop_idx']):
                dedup[j] = cand
            keep = False
            break
    if keep:
        dedup.append(cand)
print('raw total:', len(raw), '-> dedup:', len(dedup))

# Greedy selection wie im Code
remaining = sorted(dedup, key=lambda x: float(x.get('score', 1e9)))
selected = []
if len(remaining) > 0:
    selected.append(remaining.pop(0))
while len(selected) < NUM_OPENINGS and len(remaining) > 0:
    best_idx, best_obj = None, np.inf
    for i, cand in enumerate(remaining):
        sep = min(np.linalg.norm(np.asarray(cand['center']) - np.asarray(s['center'])) for s in selected)
        obj = float(cand['score']) - 0.06 * (sep / (median_edge + 1e-12))
        if obj < best_obj:
            best_obj = obj
            best_idx = i
    selected.append(remaining.pop(int(best_idx)))

print('selected loops:', len(selected))
for i, s in enumerate(selected):
    print(f'  sel {i}: size={len(s["loop_idx"])} score={float(s["score"]):.3f}')


thr=0.840 -> 8 raw candidates
thr=0.820 -> 8 raw candidates
thr=0.750 -> 8 raw candidates
raw total: 24 -> dedup: 4
selected loops: 3
  sel 0: size=149 score=-4.006
  sel 1: size=111 score=-3.862
  sel 2: size=89 score=-2.743


In [16]:
# Step 8 + 9: Voller Lauf + Endpoint-Reparaturstatus
reg.register_openings_auto_normals(
    min_loop_vertices=AUTO_MIN_LOOP_VERTICES,
    normal_dot_min=AUTO_NORMAL_DOT_MIN,
    face_dot_min=AUTO_FACE_DOT_MIN,
)

dbg = copy.deepcopy(reg.auto_registration_debug)
print('selection_mode:', dbg.get('selection_mode'))
print('cluster_thresholds:', dbg.get('cluster_thresholds'))
print('cluster_candidates_raw:', dbg.get('cluster_candidates_raw'))
print('candidate_count_dedup:', dbg.get('candidate_count_dedup'))
print('selected_sources:', dbg.get('selected_sources'))
print('opening_center_endpoint_indices_initial:', dbg.get('opening_center_endpoint_indices_initial'))
print('opening_center_endpoint_repair_used:', dbg.get('opening_center_endpoint_repair_used'))
print('opening_center_endpoint_repair_trials:', dbg.get('opening_center_endpoint_repair_trials'))
print('centreline_recovery:', dbg.get('centreline_recovery'))
print('fallback_from:', dbg.get('fallback_from'), '| fallback_reason:', dbg.get('fallback_reason'))
dbg


selection_mode: cluster_diverse
cluster_thresholds: [0.84, 0.82, 0.75]
cluster_candidates_raw: 8
candidate_count_dedup: 4
selected_sources: ['normal_graph_cluster', 'normal_graph_cluster', 'normal_graph_cluster']
opening_center_endpoint_indices_initial: [993, 2552, 15843]
opening_center_endpoint_repair_used: False
opening_center_endpoint_repair_trials: 0
centreline_recovery: None
fallback_from: None | fallback_reason: None


{'method': 'register_openings_auto_normals',
 'fallback_from': None,
 'fallback_reason': None,
 'min_loop_vertices': 16,
 'normal_dot_min': 0.84,
 'face_dot_min': 0.84,
 'anchor_endpoint_indices': [13235, 28, 11515],
 'anchor_center_index': 14528,
 'boundary_loop_count': 0,
 'cluster_thresholds': [0.84, 0.82, 0.75],
 'cluster_candidates_raw': 8,
 'candidate_count_raw': 8,
 'candidate_count_dedup': 4,
 'selection_mode': 'cluster_diverse',
 'opening_center_endpoint_indices_initial': [993, 2552, 15843],
 'opening_center_endpoint_repair_used': False,
 'opening_center_endpoint_repair_trials': 0,
 'selected_count': 3,
 'selected_sources': ['normal_graph_cluster',
  'normal_graph_cluster',
  'normal_graph_cluster'],
 'opening_sizes': [111, 149, 89],
 'endpoint_indices': [2552, 993, 15843]}

In [17]:
# Schritt 3: Face-Graph-Normalen-Cluster Kandidaten
base_thr = float(max(AUTO_NORMAL_DOT_MIN, AUTO_FACE_DOT_MIN))
cluster_thresholds = [base_thr, max(0.82, base_thr - 0.06), max(0.75, base_thr - 0.12)]
cluster_candidates = {}

for thr in cluster_thresholds:
    cands = reg._extract_normal_graph_cluster_candidates(
        normal_dot_min=float(thr),
        min_loop_vertices=int(AUTO_MIN_LOOP_VERTICES),
    )
    cands = sorted(cands, key=lambda x: float(x.get('score', 1e9)))
    cluster_candidates[thr] = cands
    print(f'thr={thr:.3f} -> {len(cands)} candidates')

viz_thr = None
for thr in cluster_thresholds:
    if len(cluster_candidates[thr]) >= NUM_OPENINGS:
        viz_thr = thr
        break
if viz_thr is None:
    viz_thr = cluster_thresholds[-1]

viz_candidates = cluster_candidates[viz_thr][:MAX_CANDIDATES_TO_SHOW]
print('visualized threshold:', viz_thr, '| shown candidates:', len(viz_candidates))


thr=0.840 -> 8 candidates
thr=0.820 -> 8 candidates
thr=0.750 -> 8 candidates
visualized threshold: 0.84 | shown candidates: 8


In [18]:
# Schritt 3 Visualisierung: beste Cluster-Kandidaten (nach score)
fig = go.Figure()
add_mesh(fig, verts, faces, color='lightgray', opacity=0.10)
palette = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']

for i, cand in enumerate(viz_candidates):
    idx = safe_indices(cand['loop_idx'], len(verts))
    if idx.size < 3:
        continue
    xyz = verts[idx]
    color = palette[i % len(palette)]
    score = float(cand.get('score', np.nan))
    add_loop(fig, xyz, name=f'cand {i} (score={score:.3f})', color=color, width=4, showlegend=True)
    c = np.asarray(cand['center'], dtype=np.float64).reshape(1, 3)
    add_points(fig, c, name=f'center {i}', color=color, size=5, showlegend=False, text=[str(i)])

fig.update_layout(
    title=f'Step 3: Normal graph cluster candidates | thr={viz_thr:.3f} | {CASE_NAME}',
    scene=dict(aspectmode='data'), width=1250, height=820
)
fig.show()


In [19]:
# Schritt 4: volle Auto-Opening-Registration laufen lassen
reg.register_openings_auto_normals(
    min_loop_vertices=AUTO_MIN_LOOP_VERTICES,
    normal_dot_min=AUTO_NORMAL_DOT_MIN,
    face_dot_min=AUTO_FACE_DOT_MIN,
)

debug = copy.deepcopy(reg.auto_registration_debug)
debug


{'method': 'register_openings_auto_normals',
 'fallback_from': None,
 'fallback_reason': None,
 'min_loop_vertices': 16,
 'normal_dot_min': 0.84,
 'face_dot_min': 0.84,
 'anchor_endpoint_indices': [13235, 28, 11515],
 'anchor_center_index': 14528,
 'boundary_loop_count': 0,
 'cluster_thresholds': [0.84, 0.82, 0.75],
 'cluster_candidates_raw': 8,
 'candidate_count_raw': 8,
 'candidate_count_dedup': 4,
 'selection_mode': 'cluster_diverse',
 'opening_center_endpoint_indices_initial': [993, 2552, 15843],
 'opening_center_endpoint_repair_used': False,
 'opening_center_endpoint_repair_trials': 0,
 'selected_count': 3,
 'selected_sources': ['normal_graph_cluster',
  'normal_graph_cluster',
  'normal_graph_cluster'],
 'opening_sizes': [111, 149, 89],
 'endpoint_indices': [2552, 993, 15843]}

In [ ]:
# Schritt 4 Visualisierung: finale ausgewaehlte Opening-Loops
fig = go.Figure()
add_mesh(fig, verts, faces, color='lightgray', opacity=0.10)

for i, idx in enumerate(reg.op_v_indices):
    idx = safe_indices(idx, len(verts))
    if idx.size < 3:
        continue
    xyz = verts[idx]
    color = 'deepskyblue'
    add_loop(fig, xyz, name=f'auto opening {i}', color=color, width=5, showlegend=(i == 0))

centers = np.vstack([np.mean(verts[safe_indices(idx, len(verts))], axis=0) for idx in reg.op_v_indices]) if len(reg.op_v_indices) else np.zeros((0,3))
add_points(fig, centers, name='opening centers', color='dodgerblue', size=6, symbol='circle', showlegend=True)

fig.update_layout(title=f'Step 4: selected auto openings | {CASE_NAME}', scene=dict(aspectmode='data'), width=1150, height=780)
fig.show()


In [ ]:
# Schritt 5: Opening-Patches bauen + Centerline/Endpunkte berechnen
reg.create_opening_meshes(viz=False)
reg.register_centreline_end_points(auto=True)

cep_idx = safe_indices(getattr(reg, 'cep_registration', []), len(verts))
if cep_idx.size >= 2:
    bif_idx = int(reg._estimate_bifurcation_index(cep_idx.tolist()))
else:
    bif_idx = None

centerline_polylines = centerline_paths_from_reg(reg, verts)
print('endpoints:', cep_idx.tolist())
print('branches:', len(centerline_polylines))


In [ ]:
# Schritt 5 Visualisierung: finale Auto-Registration (Patches + Centerline + Endpunkte)
fig = go.Figure()
add_mesh(fig, verts, faces, color='lightgray', opacity=0.10)
add_opening_surfaces_from_reg(fig, reg, color='deepskyblue', opacity=0.45)

for i, xyz in enumerate(centerline_polylines):
    fig.add_trace(go.Scatter3d(
        x=xyz[:,0], y=xyz[:,1], z=xyz[:,2],
        mode='lines', line=dict(color='cyan', width=7),
        name='auto centerline', showlegend=(i==0)
    ))

if cep_idx.size > 0:
    cep_pts = verts[cep_idx]
    add_points(
        fig, cep_pts, name='auto endpoints', color='navy', size=7,
        symbol='diamond', showlegend=True, text=[str(int(i)) for i in cep_idx]
    )

if bif_idx is not None and 0 <= bif_idx < len(verts):
    add_points(fig, verts[bif_idx].reshape(1,3), name='auto bifurcation', color='black', size=7, symbol='x', showlegend=True)

fig.update_layout(title=f'Step 5: final auto registration | {CASE_NAME}', scene=dict(aspectmode='data'), width=1200, height=820)
fig.show()


## Optional: Wichtige Debug-Felder
- `cluster_thresholds`
- `cluster_candidates_raw`
- `candidate_count_dedup`
- `selection_mode`
- `selected_sources`
- `fallback_from`
